# Ch.3 — Kubernetes Basics

> Deploy containerized applications to a self-healing, scalable cluster. Learn K8s without any cloud spend using **Kind** (Kubernetes in Docker).

**Running Example:** Flask API with 3 replicas  
**Constraint:** Local-only (no cloud costs)  
**Tech Stack:** Kind, kubectl, Docker

## The Core Idea

Kubernetes orchestrates containers across clusters of machines. Instead of running containers manually, you declare the desired state (e.g., "3 replicas of this Flask app") and K8s continuously ensures reality matches.

**Key abstractions:**
- **Pod** — smallest unit (1+ containers)
- **Deployment** — manages replicas with rolling updates
- **Service** — stable load-balanced endpoint

Training mindset: Think declaratively ("I want 3 pods") not imperatively ("start container A, B, C").

## Cell 1 — Install Kind and kubectl

We'll use **Kind** (Kubernetes in Docker) to run a full K8s cluster locally. No cloud account needed.

In [ ]:
# ── Installation Check ─────────────────────────────────────────────────────
import subprocess
import os

def check_command(cmd, install_msg):
    """Check if a command exists, print install instructions if not."""
    try:
        result = subprocess.run(
            [cmd, "version"],
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode == 0:
            version_line = result.stdout.split('\n')[0]
            print(f"✅ {cmd} installed: {version_line}")
            return True
        else:
            print(f"❌ {cmd} not found. {install_msg}")
            return False
    except FileNotFoundError:
        print(f"❌ {cmd} not found. {install_msg}")
        return False
    except Exception as e:
        print(f"⚠️  Error checking {cmd}: {e}")
        return False

# Check kubectl
kubectl_install = """
Install kubectl:
  macOS: brew install kubectl
  Linux: curl -LO "https://dl.k8s.io/release/$(curl -L -s https://dl.k8s.io/release/stable.txt)/bin/linux/amd64/kubectl"
  Windows: choco install kubernetes-cli
  Docs: https://kubernetes.io/docs/tasks/tools/
"""

# Check kind
kind_install = """
Install Kind:
  macOS: brew install kind
  Linux: curl -Lo ./kind https://kind.sigs.k8s.io/dl/latest/kind-linux-amd64 && chmod +x ./kind && sudo mv ./kind /usr/local/bin/kind
  Windows: choco install kind
  Docs: https://kind.sigs.k8s.io/docs/user/quick-start/
"""

kubectl_ok = check_command("kubectl", kubectl_install)
kind_ok = check_command("kind", kind_install)

if kubectl_ok and kind_ok:
    print("\n🎉 All prerequisites installed! Ready to create a K8s cluster.")
else:
    print("\n⚠️  Please install missing tools before proceeding.")

## Cell 2 — Create Local Kubernetes Cluster

Create a Kind cluster with a single control-plane node. This simulates a production K8s cluster on your laptop.

In [ ]:
# ── Create Kind Cluster ────────────────────────────────────────────────────
CLUSTER_NAME = "devops-demo"

# Check if cluster already exists
result = subprocess.run(
    ["kind", "get", "clusters"],
    capture_output=True,
    text=True
)

if CLUSTER_NAME in result.stdout:
    print(f"✅ Cluster '{CLUSTER_NAME}' already exists.")
    
    # Set kubectl context
    subprocess.run(["kubectl", "cluster-info", "--context", f"kind-{CLUSTER_NAME}"])
else:
    print(f"Creating Kind cluster: {CLUSTER_NAME}...")
    
    # Create cluster with custom config for port mapping
    kind_config = """
kind: Cluster
apiVersion: kind.x-k8s.io/v1alpha4
nodes:
- role: control-plane
  extraPortMappings:
  - containerPort: 30000
    hostPort: 30000
    protocol: TCP
"""
    
    config_path = "kind-config.yaml"
    with open(config_path, "w") as f:
        f.write(kind_config)
    
    result = subprocess.run(
        ["kind", "create", "cluster", "--name", CLUSTER_NAME, "--config", config_path],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print(f"✅ Cluster '{CLUSTER_NAME}' created successfully!")
        print(result.stdout)
    else:
        print(f"❌ Failed to create cluster: {result.stderr}")
    
    # Clean up config file
    if os.path.exists(config_path):
        os.remove(config_path)

# Verify cluster is ready
print("\n🔍 Checking cluster nodes:")
subprocess.run(["kubectl", "get", "nodes"])

## Cell 3 — Deploy a Pod (Single Container)

Start with the simplest unit: a single pod running Nginx. Pods are ephemeral — if killed, they won't restart automatically (that's what Deployments are for).

In [ ]:
# ── Create a Single Pod ────────────────────────────────────────────────────
pod_yaml = """
apiVersion: v1
kind: Pod
metadata:
  name: nginx-pod
  labels:
    app: nginx
spec:
  containers:
  - name: nginx
    image: nginx:1.25
    ports:
    - containerPort: 80
"""

# Write YAML to file
pod_file = "nginx-pod.yaml"
with open(pod_file, "w") as f:
    f.write(pod_yaml)

print("📄 Created nginx-pod.yaml")

# Apply to cluster
result = subprocess.run(
    ["kubectl", "apply", "-f", pod_file],
    capture_output=True,
    text=True
)
print(result.stdout)

# Wait a moment for pod to start
import time
time.sleep(3)

# Check pod status
print("\n🔍 Pod status:")
subprocess.run(["kubectl", "get", "pods", "-o", "wide"])

# Clean up YAML file
if os.path.exists(pod_file):
    os.remove(pod_file)
    
print("\n💡 Note: This pod won't restart if you kill it. For self-healing, use Deployments.")

## Cell 4 — Create a Deployment (Multiple Replicas)

Deployments manage pods with automatic restarts and replica management. This is what you'll use in production.

In [ ]:
# ── Create Deployment with 3 Replicas ──────────────────────────────────────
deployment_yaml = """
apiVersion: apps/v1
kind: Deployment
metadata:
  name: flask-deployment
  labels:
    app: flask
spec:
  replicas: 3
  selector:
    matchLabels:
      app: flask
  template:
    metadata:
      labels:
        app: flask
    spec:
      containers:
      - name: flask
        image: hashicorp/http-echo:latest
        args:
          - "-text=SmartVal API v1.0 - Pod: $(HOSTNAME)"
        ports:
        - containerPort: 5678
        env:
        - name: HOSTNAME
          valueFrom:
            fieldRef:
              fieldPath: metadata.name
"""

deployment_file = "flask-deployment.yaml"
with open(deployment_file, "w") as f:
    f.write(deployment_yaml)

print("📄 Created flask-deployment.yaml (3 replicas)")

# Apply deployment
result = subprocess.run(
    ["kubectl", "apply", "-f", deployment_file],
    capture_output=True,
    text=True
)
print(result.stdout)

# Wait for deployment to roll out
time.sleep(5)

# Check deployment status
print("\n🔍 Deployment status:")
subprocess.run(["kubectl", "get", "deployments"])

print("\n🔍 Pods created by deployment:")
subprocess.run(["kubectl", "get", "pods", "-l", "app=flask", "-o", "wide"])

# Clean up YAML file
if os.path.exists(deployment_file):
    os.remove(deployment_file)

print("\n💡 Deployment ensures 3 replicas are always running. Try deleting a pod!")

## Cell 5 — Expose Deployment as a Service

Services provide stable endpoints and load-balance across pod replicas.

In [ ]:
# ── Create Service (NodePort) ──────────────────────────────────────────────
service_yaml = """
apiVersion: v1
kind: Service
metadata:
  name: flask-service
spec:
  type: NodePort
  selector:
    app: flask
  ports:
  - port: 5678
    targetPort: 5678
    nodePort: 30000
"""

service_file = "flask-service.yaml"
with open(service_file, "w") as f:
    f.write(service_yaml)

print("📄 Created flask-service.yaml (NodePort on 30000)")

# Apply service
result = subprocess.run(
    ["kubectl", "apply", "-f", service_file],
    capture_output=True,
    text=True
)
print(result.stdout)

time.sleep(2)

# Check service status
print("\n🔍 Service status:")
subprocess.run(["kubectl", "get", "services"])

# Check endpoints (pod IPs the service routes to)
print("\n🔍 Service endpoints:")
subprocess.run(["kubectl", "get", "endpoints", "flask-service"])

# Clean up YAML file
if os.path.exists(service_file):
    os.remove(service_file)

print("\n💡 Service 'flask-service' load-balances across 3 pods.")
print("   Access it at: http://localhost:30000")

## Cell 6 — Access Service from Host Machine

Test the service by making HTTP requests. The load balancer distributes traffic across replicas.

In [ ]:
# ── Test Service Endpoint ──────────────────────────────────────────────────
import requests
import time

SERVICE_URL = "http://localhost:30000"

print(f"🌐 Testing service at {SERVICE_URL}\n")

try:
    # Make multiple requests to see load balancing
    for i in range(6):
        response = requests.get(SERVICE_URL, timeout=5)
        print(f"Request {i+1}: {response.text.strip()}")
        time.sleep(0.5)
    
    print("\n✅ Service is accessible! Notice requests hit different pods (load balancing).")
except requests.exceptions.ConnectionError:
    print("❌ Connection failed. Ensure the service is running:")
    print("   kubectl get svc flask-service")
    print("   kubectl get pods -l app=flask")
except Exception as e:
    print(f"❌ Error: {e}")

## Cell 7 — Rolling Updates (Change Image Version)

Update the deployment to a new version. K8s gradually replaces old pods with new ones (zero downtime).

In [ ]:
# ── Perform Rolling Update ─────────────────────────────────────────────────
print("🔄 Updating deployment to v1.1 (simulated by changing environment variable)...")

# Update deployment image using kubectl set
result = subprocess.run(
    ["kubectl", "set", "env", "deployment/flask-deployment", "VERSION=v1.1"],
    capture_output=True,
    text=True
)
print(result.stdout)

# Watch rollout status
print("\n🔍 Watching rollout progress:")
subprocess.run(["kubectl", "rollout", "status", "deployment/flask-deployment"])

# Check new pods
print("\n🔍 Updated pods:")
subprocess.run(["kubectl", "get", "pods", "-l", "app=flask"])

# Check rollout history
print("\n📜 Rollout history:")
subprocess.run(["kubectl", "rollout", "history", "deployment/flask-deployment"])

print("\n✅ Rolling update complete! Old pods replaced gradually (zero downtime).")

## Cell 8 — Rollback a Deployment

If the new version has bugs, rollback to the previous version instantly.

In [ ]:
# ── Rollback to Previous Version ───────────────────────────────────────────
print("⏪ Rolling back to previous version...")

# Undo last rollout
result = subprocess.run(
    ["kubectl", "rollout", "undo", "deployment/flask-deployment"],
    capture_output=True,
    text=True
)
print(result.stdout)

# Watch rollback progress
print("\n🔍 Watching rollback progress:")
subprocess.run(["kubectl", "rollout", "status", "deployment/flask-deployment"])

# Verify rollback
print("\n🔍 Pods after rollback:")
subprocess.run(["kubectl", "get", "pods", "-l", "app=flask"])

print("\n✅ Rollback complete! Deployment restored to previous state.")

## Cell 9 — ConfigMaps and Secrets

Store configuration data separately from code. ConfigMaps for non-sensitive data, Secrets for passwords/tokens.

In [ ]:
# ── Create ConfigMap and Secret ────────────────────────────────────────────

# Create ConfigMap
configmap_yaml = """
apiVersion: v1
kind: ConfigMap
metadata:
  name: app-config
data:
  MODEL_PATH: "/models/smartval-v1.pkl"
  LOG_LEVEL: "INFO"
  MAX_WORKERS: "4"
"""

# Create Secret (base64 encoded)
import base64
api_key_encoded = base64.b64encode(b"sk-demo-key-12345").decode()

secret_yaml = f"""
apiVersion: v1
kind: Secret
metadata:
  name: app-secrets
type: Opaque
data:
  API_KEY: {api_key_encoded}
"""

# Write and apply ConfigMap
config_file = "configmap.yaml"
with open(config_file, "w") as f:
    f.write(configmap_yaml)

subprocess.run(["kubectl", "apply", "-f", config_file], capture_output=True)
print("✅ ConfigMap created: app-config")

# Write and apply Secret
secret_file = "secret.yaml"
with open(secret_file, "w") as f:
    f.write(secret_yaml)

subprocess.run(["kubectl", "apply", "-f", secret_file], capture_output=True)
print("✅ Secret created: app-secrets")

# Show created resources
print("\n🔍 ConfigMaps:")
subprocess.run(["kubectl", "get", "configmaps"])

print("\n🔍 Secrets:")
subprocess.run(["kubectl", "get", "secrets"])

# Clean up YAML files
for f in [config_file, secret_file]:
    if os.path.exists(f):
        os.remove(f)

print("\n💡 Use these in deployments via 'envFrom' or 'volumeMounts'.")
print("   ConfigMaps: non-sensitive config (model paths, feature flags)")
print("   Secrets: passwords, API keys, certificates")

## Cell 10 — Debugging with kubectl logs and describe

Essential commands for troubleshooting pods that crash or won't start.

In [ ]:
# ── Debugging Commands ─────────────────────────────────────────────────────

print("🛠️  Essential Kubernetes debugging commands:\n")

# 1. Get all pods with status
print("1️⃣ View all pods (with status):")
print("   kubectl get pods -o wide\n")
subprocess.run(["kubectl", "get", "pods", "-A", "-o", "wide"])

# 2. Describe a pod (detailed events)
print("\n2️⃣ Describe a specific pod (shows events, restart count, etc.):")
print("   kubectl describe pod <pod-name>")

# Get first flask pod
result = subprocess.run(
    ["kubectl", "get", "pods", "-l", "app=flask", "-o", "jsonpath={.items[0].metadata.name}"],
    capture_output=True,
    text=True
)
pod_name = result.stdout.strip()

if pod_name:
    print(f"\n   Example: kubectl describe pod {pod_name}")
    subprocess.run(["kubectl", "describe", "pod", pod_name])

# 3. View logs
print("\n3️⃣ View pod logs:")
print(f"   kubectl logs {pod_name}")

if pod_name:
    result = subprocess.run(
        ["kubectl", "logs", pod_name, "--tail=10"],
        capture_output=True,
        text=True
    )
    print(f"\n   Last 10 lines from {pod_name}:")
    print(result.stdout)

# 4. Exec into pod
print("\n4️⃣ SSH into a running pod:")
print(f"   kubectl exec -it {pod_name} -- /bin/sh")

# 5. Check events
print("\n5️⃣ View recent cluster events:")
print("   kubectl get events --sort-by=.metadata.creationTimestamp --all-namespaces")

print("\n" + "="*70)
print("📋 Debugging Workflow:")
print("  1. kubectl get pods          → Check status (Running, Pending, CrashLoopBackOff)")
print("  2. kubectl describe pod      → Detailed events and resource info")
print("  3. kubectl logs <pod>        → Application stdout/stderr")
print("  4. kubectl exec -it <pod>    → Interactive shell inside container")
print("  5. kubectl get events        → Cluster-wide event history")
print("="*70)

print("\n✅ You now have the essential tools for K8s debugging!")

## Summary

You've deployed a self-healing, load-balanced application to a local Kubernetes cluster:

✅ **Created a Kind cluster** — full K8s environment on your laptop  
✅ **Deployed pods and deployments** — understood the difference  
✅ **Exposed services** — stable load-balanced endpoints  
✅ **Performed rolling updates** — zero-downtime deployments  
✅ **Rolled back** — instant recovery from bad deployments  
✅ **Used ConfigMaps/Secrets** — separated config from code  
✅ **Learned debugging tools** — `kubectl logs`, `describe`, `exec`

**Next:** Ch.4 (CI/CD Pipelines) automates this entire workflow — every push to `main` triggers a deployment to K8s.

**Cleanup:**
```bash
# Delete cluster when done
kind delete cluster --name devops-demo
```